# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies


## Task 2: Environment Variables

We'll want to set our OpenAI, Tavily, and LangSmith API keys along with our LangSmith environment variables.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE8 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

# Configure Tavily search tool with optimized settings
tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced",  # Use advanced search for better results
    include_answer=True,       # Include direct answers when available
    include_raw_content=False # Don't include raw content to keep responses concise
)

# Configure Arxiv tool for academic paper search
arxiv_tool = ArxivQueryRun(
    max_results=5,  # Limit results for better performance
    doc_content_chars_max=2000  # Limit content length
)

# Create our toolbelt with properly configured tools
tool_belt = [
    tavily_tool,
    arxiv_tool,
]

# Display tool information for verification
print("🔧 Toolbelt Configuration:")
print(f"📊 Total tools: {len(tool_belt)}")
for i, tool in enumerate(tool_belt, 1):
    print(f"  {i}. {tool.name}: {tool.description[:100]}...")
print("\n✅ Toolbelt ready for use!")

🔧 Toolbelt Configuration:
📊 Total tools: 2
  1. tavily_search_results_json: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need...
  2. arxiv: A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, ...

✅ Toolbelt ready for use!


/var/folders/2n/cyq8dv9j683_ynffwh1tw3_c0000gn/T/ipykernel_84244/884237190.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [5]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [6]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

**Answer:**

The model determines which tool to use through a sophisticated process called **function calling** (also known as tool calling). Here's how it works:

## 🔍 **The Decision Process:**

### 1. **Tool Schema Understanding**
When we bind tools to the model with `model.bind_tools(tool_belt)`, the model receives:
- **Tool names** (e.g., "tavily_search_results_json", "arxiv")
- **Tool descriptions** (what each tool does)
- **Parameter schemas** (what inputs each tool expects)

### 2. **Context Analysis**
The model analyzes:
- **User query intent** - What is the user trying to accomplish?
- **Available tools** - Which tools can help answer this query?
- **Tool capabilities** - What can each tool do based on their descriptions?

### 3. **Tool Selection Logic**
The model uses several factors to decide:

**🎯 Query-Tool Matching:**
- **Search queries** → Tavily Search Tool
- **Academic/research questions** → Arxiv Tool
- **General knowledge** → May use both or neither

**📊 Confidence Scoring:**
- The model internally scores how well each tool matches the query
- Considers tool descriptions, parameter requirements, and expected outputs

**🔄 Multi-tool Strategy:**
- Can select multiple tools for complex queries
- Determines the order of tool execution
- May use tools sequentially or in parallel

### 4. **Output Format**
When the model decides to use a tool, it outputs:
```json
{
  "tool_calls": [
    {
      "name": "tavily_search_results_json",
      "args": {"query": "user's search term"},
      "id": "unique_call_id"
    }
  ]
}
```

### 5. **Conditional Logic**
The `should_continue` function checks:
- If `last_message.tool_calls` exists → Route to "action" node
- If no tool calls → Route to "END"

## 🧠 **Why This Works:**

1. **Training on Function Calling**: Modern LLMs are trained on function calling patterns
2. **Schema Understanding**: Models learn to map queries to appropriate tool schemas
3. **Context Awareness**: The model considers conversation history and user intent
4. **Error Handling**: Can retry with different tools if initial choice fails

## 🔧 **In Our Implementation:**

- **Tavily Tool**: Used for web searches, current events, general information
- **Arxiv Tool**: Used for academic papers, research questions, scientific topics
- **Model Intelligence**: Decides based on query type, keywords, and context

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [8]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [9]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [10]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [11]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [12]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [13]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

**Answer:**

## 🔄 **Default Cycling Behavior:**

**No Built-in Limit**: By default, LangGraph has **no specific limit** on the number of cycles. The graph will continue cycling until:
- The model decides not to use any tools (no `tool_calls` in the response)
- An error occurs
- The graph reaches an `END` node

## ⚠️ **Why Limits Are Important:**

Without limits, you could encounter:
- **Infinite loops** - Agent gets stuck in repetitive cycles
- **High costs** - Each cycle consumes API tokens
- **Poor user experience** - Long wait times
- **Resource exhaustion** - Memory and processing limits

## 🛡️ **Methods to Impose Cycle Limits:**

### **1. Message Count Limit** (Most Common)
```python
def should_continue_with_limit(state):
    last_message = state["messages"][-1]
    
    # Check if we've exceeded the message limit
    if len(state["messages"]) > 10:  # Limit to 10 messages
        return "END"
    
    if last_message.tool_calls:
        return "action"
    
    return "END"
```

### **2. Iteration Counter** (More Precise)
```python
class AgentStateWithCounter(TypedDict):
    messages: Annotated[list, add_messages]
    iteration_count: int

def should_continue_with_counter(state):
    last_message = state["messages"][-1]
    
    # Check iteration limit
    if state.get("iteration_count", 0) >= 5:  # Max 5 iterations
        return "END"
    
    if last_message.tool_calls:
        return "action"
    
    return "END"
```

### **3. Time-based Limits**
```python
import time

def should_continue_with_time(state):
    start_time = state.get("start_time", time.time())
    current_time = time.time()
    
    # Limit to 30 seconds
    if current_time - start_time > 30:
        return "END"
    
    # ... rest of logic
```

### **4. Cost-based Limits**
```python
def should_continue_with_cost(state):
    total_tokens = state.get("total_tokens", 0)
    
    # Limit to 10,000 tokens
    if total_tokens > 10000:
        return "END"
    
    # ... rest of logic
```

## 🔧 **Implementation in Our Graph:**

The notebook already shows an example with message count limiting:

```python
def tool_call_or_helpful(state):
    # ... existing logic ...
    
    if len(state["messages"]) > 10:  # 🛡️ Cycle limit!
        return "END"
    
    # ... rest of logic
```

## 📊 **Best Practices for Cycle Limits:**

1. **Start Conservative**: Begin with 5-10 cycles
2. **Monitor Performance**: Track completion rates and user satisfaction
3. **Adjust Based on Use Case**: Research tasks may need more cycles
4. **Add Graceful Degradation**: Provide partial results when limits are reached
5. **Log Cycle Usage**: Monitor how many cycles are typically needed

## 🎯 **Recommended Approach:**

For most applications, use **message count limiting** (10-15 messages) as it's:
- Simple to implement
- Easy to understand
- Provides good balance of functionality vs. safety
- Works well with the existing LangGraph architecture

## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [14]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="How are technical professionals using AI to improve their work?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='Technical professionals are using AI in various ways to enhance their work, including automating repetitive tasks, improving decision-making, analyzing large datasets, developing new products and services, and optimizing processes. They leverage AI for tasks such as machine learning model development, natural language processing, computer vision, and predictive analytics to increase efficiency, accuracy, and innovation in their respective fields. Would you like specific examples from particular industries or roles?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 163, 'total_tokens': 247, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [15]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the A Comprehensive Survey of Deep Research paper, then search each of the authors to find out where they work now using Tavily!")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ZhBOjnYdjCohDBeCMvMfsXJk', 'function': {'arguments': '{"query": "A Comprehensive Survey of Deep Research"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_DCPwK1mWTT1Jy5wg0j0aeFOi', 'function': {'arguments': '{"query": "author of A Comprehensive Survey of Deep Research"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 182, 'total_tokens': 242, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CLIebzoSXBwHuJONLhoszn3TJIaSh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c7cf87b5-f04

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

**Answer:**

## 🔍 **Agent Execution Steps Analysis:**

Based on the output from the query: *"Search Arxiv for the A Comprehensive Survey of Deep Research paper, then search each of the authors to find out where they work now using Tavily!"*

### **Step 1: Initial Query Processing** 🧠
- **Node**: `agent`
- **Action**: The model analyzed the complex query and identified it required multiple tools
- **Decision**: Determined it needed to use both Arxiv and Tavily tools
- **Output**: Generated tool calls for both tools simultaneously:
  - `arxiv` tool with query: "A Comprehensive Survey of Deep Research"
  - `tavily_search_results_json` tool with query: "author of A Comprehensive Survey of Deep Research"

### **Step 2: Tool Execution** 🔧
- **Node**: `action`
- **Tool Used**: `arxiv`
- **Action**: Searched Arxiv database for the paper
- **Result**: Found the paper with details:
  - **Published**: 2025-06-14
  - **Title**: "A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications"
  - **Authors**: Renjun Xu, Jingwen Peng
  - **Summary**: Detailed information about Deep Research systems, AI-powered applications, and comprehensive taxonomy

### **Step 3: Final Response Generation** 📝
- **Node**: `agent`
- **Action**: Processed the Arxiv results and attempted to search for author affiliations
- **Challenge**: Encountered SSL certificate verification error with Tavily search
- **Response**: Provided comprehensive information about the paper while acknowledging the technical limitation

## 🔄 **Graph Flow Summary:**

```
User Query → Agent Node → Action Node (Arxiv) → Agent Node → END
     ↓           ↓              ↓                ↓
  Analysis   Tool Calls    Tool Execution   Final Response
```

## 📊 **Key Observations:**

1. **Parallel Tool Planning**: The agent planned to use both tools simultaneously
2. **Successful Arxiv Search**: Successfully retrieved the research paper details
3. **Technical Limitation**: Encountered SSL error with Tavily search
4. **Graceful Handling**: Acknowledged the limitation and provided available information
5. **Single Cycle**: The process completed in one cycle (no infinite loops)

## 🎯 **Agent Intelligence Demonstrated:**

- **Query Decomposition**: Broke down complex query into actionable steps
- **Tool Selection**: Chose appropriate tools for different aspects of the query
- **Error Handling**: Gracefully handled technical limitations
- **Information Synthesis**: Combined results into coherent response
- **User Communication**: Clearly explained what was accomplished and what wasn't possible



# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [16]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["text"])]}

def parse_output(input_state):
  return {"answer" : input_state["messages"][-1].content}

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

agent_chain_with_formatting.invoke({"text" : "What is Deep Research?"})

{'answer': "Deep Research is an advanced AI-powered agent designed to conduct comprehensive, multi-step research on the Internet. Unlike standard chatbots that provide quick, surface-level responses, Deep Research autonomously finds, analyzes, and synthesizes information from hundreds of online sources to produce detailed and structured reports. It is particularly useful for tasks that require integrating multiple information sources, deep analysis of complex data, and generating well-documented outputs. OpenAI's Deep Research leverages models optimized for web browsing and data analysis, enabling it to perform complex research tasks efficiently and reliably."}

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    {
        "inputs" : {"text" : "Who were the main authors on the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' paper?"},
        "outputs" : {"must_mention" : ["Peng", "Xu"]}   
    },
    ...,
    {
        "inputs" : {"text" : "Where do the authors of the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' work now?"},
        "outputs" : {"must_mention" : ["Zhejiang", "Liberty Mutual"]}
    }
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions that pertain to the cohort use-case (more information [here](https://www.notion.so/Session-4-RAG-with-LangGraph-OSS-Local-Models-Eval-w-LangSmith-26acd547af3d80838d5beba464d7e701#26acd547af3d81d08809c9c82a462bdd)), or the use-case you're hoping to tackle in your Demo Day project.

In [17]:
questions = [
    {
        "inputs": {"text": "What are the key components of a Deep Research application using LangChain and LangGraph?"},
        "outputs": {"must_mention": ["LangChain", "LangGraph", "RAG", "vector database", "embeddings", "chains", "graphs", "nodes", "edges"]}
    },
    {
        "inputs": {"text": "What are the main differences between LangChain and LangGraph for building AI applications?"},
        "outputs": {"must_mention": ["chains", "cycles", "loops", "agent-forward", "stateful", "nodes", "edges", "RAG pipeline"]}
    },
    {
        "inputs": {"text": "What are the essential steps in a RAG pipeline for a Deep Research application?"},
        "outputs": {"must_mention": ["load documents", "split text", "embed", "vector database", "retrieve", "augment prompt", "generate answer"]}
    },
    {
        "inputs": {"text": "What are the three main patterns in Generative AI that are relevant for building AI projects?"},
        "outputs": {"must_mention": ["Context Engineering", "Fine-tuning", "Agents", "LLM-based agents"]}
    },
    {
        "inputs": {"text": "What open-source models and tools are recommended for building a Deep Research application?"},
        "outputs": {"must_mention": ["Ollama", "gpt-oss", "EmbeddingGemma", "LangSmith", "open-source models"]}
    },
    {
        "inputs": {"text": "How does LangGraph support cyclic behavior in AI applications compared to traditional linear chains?"},
        "outputs": {"must_mention": ["cycles", "loops", "agent-forward", "stateful", "conditional edges", "END nodes"]}
    },
    {
        "inputs": {"text": "What are the key benefits of using LangGraph for building complex AI workflows?"},
        "outputs": {"must_mention": ["readable", "maintainable", "agent-forward", "cyclic behavior", "stateful", "composable"]}
    },
    {
        "inputs": {"text": "What evaluation and monitoring tools are available for LangChain and LangGraph applications?"},
        "outputs": {"must_mention": ["LangSmith", "monitoring", "evaluation", "visibility", "OpenEvals", "correctness"]}
    }
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [18]:
from langsmith import Client

client = Client()

dataset_name = f"Simple Search Agent - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the cohort use-case to evaluate the Simple Search Agent."
)

client.create_examples(
    dataset_id=dataset.id,
    examples=questions
)

{'example_ids': ['9a349c16-dfc0-4d4c-9073-90266cc176b6',
  'a7cafd59-0bb5-46b4-8e47-f9ef23ec2180',
  'dda59ed0-0f5d-4831-8249-35d0c564a22f',
  'd06b3659-c33c-4548-b919-0628e5e3fa22',
  '9e260e4a-2dd7-4f8a-9fd3-d22a5eea8e79',
  '83efa2f9-6911-4cd7-8076-a552dbb4a72e',
  'c185788a-a900-4539-b4bc-c25f30dfa1e4',
  'ab515997-0335-41d6-9272-ea1e1ca820cf'],
 'count': 8}

### Task 2: Adding Evaluators

Let's use the OpenEvals library to product an evaluator that we can then pass into LangSmith!

> NOTE: Examine the `CORRECTNESS_PROMPT` below!

In [19]:
from openevals.prompts import CORRECTNESS_PROMPT
print(CORRECTNESS_PROMPT)

You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct answer:
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
  - Uses precise and accurate terminology

  When scoring, you should penalize:
  - Factual errors or inaccuracies
  - Incomplete or partial answers
  - Misleading or ambiguous statements
  - Incorrect terminology
  - Logical inconsistencies
  - Missing key information
</Rubric>

<Instructions>
  - Carefully read the input and output
  - Check for factual accuracy and completeness
  - Focus on correctness of information rather than style or verbosity
</Instructions>

<Reminder>
  The goal is to evaluate factual correctness and completeness of the response.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

Use the reference outputs below to help you evaluate the

In [20]:
from openevals.llm import create_llm_as_judge

correctness_evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="openai:o3-mini", # very impactful to the final score
        feedback_key="correctness",
    )

Let's also create a custom Evaluator for our created dataset above - we do this by first making a simple Python function!

In [21]:
def must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
  # determine if the phrases in the reference_outputs are in the outputs
  required = reference_outputs.get("must_mention") or []
  score = all(phrase in outputs["answer"] for phrase in required)
  return score

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

**Answer:**

## 🔍 **Current Metric Analysis:**

The current `must_mention` metric has several limitations:

```python
def must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    score = all(phrase in outputs["answer"] for phrase in required)
    return score
```

## ⚠️ **Current Limitations:**

1. **Binary Scoring**: Only returns 0 or 1, no partial credit
2. **Case Sensitivity**: Fails if capitalization differs
3. **Exact Matching**: Misses synonyms, variations, or related terms
4. **No Context Awareness**: Doesn't consider if terms are used correctly
5. **No Weighting**: All terms treated equally important
6. **No Negation Handling**: Doesn't penalize incorrect information

## 🚀 **Improvement Strategies:**

### **1. Partial Credit System**
```python
def must_mention_improved(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    found_terms = []
    for phrase in required:
        if phrase.lower() in outputs["answer"].lower():
            found_terms.append(phrase)
    
    return len(found_terms) / len(required)  # Partial credit
```

### **2. Fuzzy Matching with Synonyms**
```python
import difflib
from collections import defaultdict

def must_mention_fuzzy(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    answer = outputs["answer"].lower()
    
    # Define synonyms and related terms
    synonyms = {
        "LangChain": ["langchain", "lang chain", "langchain framework"],
        "RAG": ["retrieval augmented generation", "rag pipeline", "retrieval-augmented"],
        "vector database": ["vector db", "vector store", "embedding database"]
    }
    
    score = 0
    for phrase in required:
        phrase_lower = phrase.lower()
        if phrase_lower in answer:
            score += 1
        else:
            # Check synonyms
            for key, values in synonyms.items():
                if key.lower() == phrase_lower:
                    for synonym in values:
                        if synonym in answer:
                            score += 0.8  # Partial credit for synonyms
                            break
    
    return score / len(required)
```

### **3. Weighted Importance Scoring**
```python
def must_mention_weighted(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    weights = reference_outputs.get("weights", [1.0] * len(required))
    
    total_weight = sum(weights)
    earned_weight = 0
    
    for i, phrase in enumerate(required):
        if phrase.lower() in outputs["answer"].lower():
            earned_weight += weights[i]
    
    return earned_weight / total_weight
```

### **4. Context-Aware Scoring**
```python
import re

def must_mention_contextual(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    answer = outputs["answer"]
    
    score = 0
    for phrase in required:
        # Check for exact match
        if phrase.lower() in answer.lower():
            score += 1
        else:
            # Check for related concepts
            related_patterns = {
                "LangChain": r"langchain|lang\s*chain|langchain\s+framework",
                "RAG": r"retrieval\s+augmented\s+generation|rag\s+pipeline",
                "vector database": r"vector\s+(db|database|store|index)"
            }
            
            if phrase in related_patterns:
                if re.search(related_patterns[phrase], answer, re.IGNORECASE):
                    score += 0.7  # Partial credit for related concepts
    
    return score / len(required)
```

### **5. Multi-Dimensional Evaluation**
```python
def must_mention_comprehensive(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    required = reference_outputs.get("must_mention") or []
    answer = outputs["answer"]
    
    # Multiple scoring dimensions
    scores = {
        "completeness": 0,  # How many required terms mentioned
        "accuracy": 0,      # Are terms used correctly
        "relevance": 0,      # Are terms relevant to the question
        "clarity": 0         # Are terms explained clearly
    }
    
    # Completeness scoring
    found_terms = [term for term in required if term.lower() in answer.lower()]
    scores["completeness"] = len(found_terms) / len(required)
    
    # Accuracy scoring (check for correct usage)
    for term in found_terms:
        # Look for context around the term
        term_context = re.search(rf".{{0,50}}{re.escape(term)}.{{0,50}}", answer, re.IGNORECASE)
        if term_context:
            context = term_context.group().lower()
            # Check if term is used in appropriate context
            if any(keyword in context for keyword in ["framework", "tool", "system", "application"]):
                scores["accuracy"] += 1
    
    scores["accuracy"] = scores["accuracy"] / len(found_terms) if found_terms else 0
    
    # Weighted final score
    weights = {"completeness": 0.4, "accuracy": 0.3, "relevance": 0.2, "clarity": 0.1}
    final_score = sum(scores[dim] * weights[dim] for dim in scores)
    
    return final_score
```

## 🎯 **Recommended Implementation:**

For production use, I recommend combining approaches:

```python
def must_mention_production(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
    """Production-ready metric with multiple improvements"""
    required = reference_outputs.get("must_mention") or []
    if not required:
        return 1.0
    
    answer = outputs["answer"].lower()
    found_terms = []
    
    for phrase in required:
        phrase_lower = phrase.lower()
        if phrase_lower in answer:
            found_terms.append(phrase)
        else:
            # Check for common variations
            variations = [
                phrase_lower.replace(" ", ""),
                phrase_lower.replace("-", " "),
                phrase_lower.replace("_", " ")
            ]
            if any(var in answer for var in variations):
                found_terms.append(phrase)
    
    # Return partial credit
    return len(found_terms) / len(required)
```

## 📊 **Key Improvements:**

1. **Partial Credit**: Rewards partial completion
2. **Case Insensitive**: Handles different capitalizations
3. **Fuzzy Matching**: Catches variations and synonyms
4. **Weighted Scoring**: Prioritizes important terms
5. **Context Awareness**: Considers how terms are used
6. **Multi-dimensional**: Evaluates completeness, accuracy, relevance
7. **Production Ready**: Robust error handling and edge cases

Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [22]:
results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset.name,
    evaluators=[correctness_evaluator, must_mention],
    experiment_prefix="simple_agent, baseline",  # optional, experiment name prefix
    description="Testing the baseline system.",  # optional, experiment description
    max_concurrency=4, # optional, add concurrency
)

View the evaluation results for experiment: 'simple_agent, baseline-4ea87497' at:
https://smith.langchain.com/o/77ce7a24-10c1-4621-b352-72f530436ef8/datasets/593ea45a-6c82-49b7-947c-e0bd7eef23fe/compare?selectedSessions=1e737efa-5155-42ba-94dc-f678d17bfe05




0it [00:00, ?it/s]

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [23]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #4:

Please write markdown for the following cells to explain what each is doing.

##### YOUR MARKDOWN HERE

In [ ]:
# Create a completely new graph instance to avoid conflicts
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### **Graph Initialization and Node Setup**

This cell creates a new LangGraph instance specifically for the helpfulness check functionality. It:

- **Creates a new StateGraph**: Uses the same `AgentState` structure as the original graph
- **Adds the agent node**: Connects the `call_model` function to handle LLM interactions
- **Adds the action node**: Connects the `tool_node` to execute tool calls
- **Avoids conflicts**: Creates a completely separate graph instance to prevent interference with the original simple agent

This setup ensures we have a clean slate for implementing the enhanced helpfulness checking functionality.

In [21]:
graph_with_helpfulness_check.set_entry_point("agent")

##### **Setting the Entry Point**

This cell establishes the starting point for our enhanced graph:

- **Sets entry point to "agent"**: The graph will always begin with the agent node
- **Consistent with original**: Maintains the same entry point as the simple agent
- **Agent-first approach**: Ensures the LLM gets the first opportunity to process the user query
- **Foundation for routing**: Sets up the initial state for the conditional logic that follows

This entry point ensures that every conversation starts with the agent analyzing the user's request before deciding on the next steps.

##### **Conditional Edge Configuration**

This cell connects the agent node to the appropriate next steps based on the conditional logic:

- **Routes to "continue"**: When the response is unhelpful, loops back to the agent for another attempt
- **Routes to "action"**: When the agent wants to use tools, sends the request to the tool node
- **Routes to "end"**: When the response is helpful or cycle limit is reached, terminates the conversation
- **Three-way routing**: Provides flexible decision-making based on multiple criteria
- **Prevents infinite loops**: The "end" route ensures the conversation can terminate

This configuration enables the agent to intelligently decide whether to continue processing, use tools, or conclude the conversation based on the quality and completeness of its responses.


In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

##### **Tool-to-Agent Return Path**

This cell establishes the return path from tool execution back to the agent:

- **Action to agent edge**: After tools are executed, the results are sent back to the agent
- **Enables tool result processing**: The agent can analyze tool outputs and decide next steps
- **Maintains conversation flow**: Ensures tool results are properly integrated into the response
- **Supports iterative tool use**: Allows the agent to use multiple tools in sequence
- **Completes the cycle**: Forms the complete loop from agent → tools → agent

This edge is essential for the agent to process tool results and continue the conversation flow, whether that means using more tools or providing a final response.


##### **Enhanced Conditional Logic with Helpfulness Check**

This cell implements the sophisticated `tool_call_or_helpful` function that adds intelligent decision-making:

- **Tool call detection**: Checks if the agent wants to use tools (`last_message.tool_calls`)
- **Cycle limit protection**: Prevents infinite loops by limiting to 10 messages
- **Helpfulness evaluation**: Uses a separate LLM to assess if the response adequately answers the user's question
- **Multi-criteria decision**: Routes based on tool calls, cycle limits, and helpfulness assessment
- **Intelligent routing**: Returns "action" for tool calls, "continue" for unhelpful responses, or "end" for helpful responses

This function represents the core intelligence of the enhanced agent, combining tool usage, cycle management, and response quality assessment.

##### **Graph Compilation and Finalization**

This cell compiles the enhanced graph into an executable application:

- **Compiles the graph**: Converts the graph definition into a runnable application
- **Enables execution**: Makes the graph ready for processing user queries
- **Integrates all components**: Combines nodes, edges, and conditional logic into a cohesive system
- **Creates the final agent**: Produces the `agent_with_helpfulness_check` that can be used for conversations
- **Ready for deployment**: The compiled graph is now ready to handle real user interactions

This step transforms our graph definition into a functional agent that can intelligently process queries, use tools, and provide helpful responses while managing conversation flow and quality.


In [23]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### **Conditional Edge Configuration**

This cell connects the agent node to the appropriate next steps based on the conditional logic:

- **Routes to "continue"**: When the response is unhelpful, loops back to the agent for another attempt
- **Routes to "action"**: When the agent wants to use tools, sends the request to the tool node
- **Routes to "end"**: When the response is helpful or cycle limit is reached, terminates the conversation
- **Three-way routing**: Provides flexible decision-making based on multiple criteria
- **Prevents infinite loops**: The "end" route ensures the conversation can terminate

This configuration enables the agent to intelligently decide whether to continue processing, use tools, or conclude the conversation based on the quality and completeness of its responses.

In [24]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### **Tool-to-Agent Return Path**

This cell establishes the return path from tool execution back to the agent:

- **Action to agent edge**: After tools are executed, the results are sent back to the agent
- **Enables tool result processing**: The agent can analyze tool outputs and decide next steps
- **Maintains conversation flow**: Ensures tool results are properly integrated into the response
- **Supports iterative tool use**: Allows the agent to use multiple tools in sequence
- **Completes the cycle**: Forms the complete loop from agent → tools → agent

This edge is essential for the agent to process tool results and continue the conversation flow, whether that means using more tools or providing a final response.

In [25]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### **Graph Compilation and Finalization**

This cell compiles the enhanced graph into an executable application:

- **Compiles the graph**: Converts the graph definition into a runnable application
- **Enables execution**: Makes the graph ready for processing user queries
- **Integrates all components**: Combines nodes, edges, and conditional logic into a cohesive system
- **Creates the final agent**: Produces the `agent_with_helpfulness_check` that can be used for conversations
- **Ready for deployment**: The compiled graph is now ready to handle real user interactions

This step transforms our graph definition into a functional agent that can intelligently process queries, use tools, and provide helpful responses while managing conversation flow and quality.

In [27]:
inputs = {"messages" : [HumanMessage(content="What are Deep Research Agents?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='Deep Research Agents are advanced AI systems designed to assist with in-depth research tasks. They leverage deep learning techniques and large datasets to analyze complex information, generate insights, and support decision-making across various fields such as science, technology, medicine, and more. These agents can automate literature reviews, extract relevant data from vast sources, and provide comprehensive summaries, making research more efficient and thorough. Would you like me to find more detailed or specific information about Deep Research Agents?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 158, 'total_tokens': 251, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-

## Part 3: LangGraph for the "Patterns" of GenAI

### Task 4: Helpfulness Check of Gen AI Pattern Descriptions

Let's ask our system about the 3 main patterns in Generative AI:

1. Context Engineering
2. Fine-tuning
3. Agents

In [28]:
patterns = ["Context Engineering", "Fine-tuning", "LLM-based agents"]

In [29]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

Context Engineering is a relatively new interdisciplinary field that focuses on designing, managing, and optimizing the context in which systems, especially artificial intelligence and software applications, operate. It involves understanding and shaping the environment, circumstances, and background information that influence how systems behave and interact with users. The goal is to improve system performance, user experience, and decision-making by carefully engineering the context.

The concept of Context Engineering has gained prominence with the rise of AI, IoT, and complex software systems, where context plays a crucial role in system effectiveness. It started to break onto the scene in the late 2010s and early 2020s, driven by advancements in contextual AI, ubiquitous computing, and the need for more adaptive and personalized systems.

Would you like me to find more detailed or specific information about its origins and development?



Fine-tuning is a machine learning techniqu